# Measurments


Jakob:

* Knee to hip: 44 cm
* hip to hip: 25 cm
* knee to ankle: 43 cm
* hip to sholder: 50 cm
* shoulder to shoulder: 38 cm
* movement: 65 cm

Enis: 

* shoulder_to_shoulder = 0.036
* Shoulder_to_hip = 0.55
* hip_to_hip = 0.25
* hip_to_knee = 0.47
* knee_to_ankle = 0.43

## Data Augmentation

Three augmentation techniques were implemented to improve model generalization and reduce overfitting:

### 1. Mirror Augmentation (Y-Plane)
- Mirrors poses left-right by swapping and negating x-coordinates of symmetric joints
- Doubles the effective training dataset size
- **Result**: 3.6% improvement in validation MAE (0.0446 → 0.0430)

### 2. Rotation Augmentation (Around Y-Axis)
- Rotates the entire pose around the vertical (Y) axis by small angles
- Helps model become invariant to camera angle changes
- **Status**: Implemented but computationally expensive

### 3. Noise Augmentation
- Adds Gaussian noise to joint coordinates
- Smaller noise variance for Z-depth (0.001) than X/Y (0.002)
- Makes model robust to minor detection errors
- **Status**: Implemented but not tested, nodes could already have alot of noise


## Sequencing data

### Data Preparation

The time-series data was sequenced using a sliding window approach to capture temporal dependencies in pose movements. Each sequence contains `seq_length` consecutive frames, creating input-output pairs where:

- **Input (X)**: 2D joint coordinates (x, y) for 13 joints over `seq_length` frames → shape: `(seq_length, 26)`
- **Target (Y)**: 3D depth coordinates (z) for 13 joints over `seq_length` frames → shape: `(seq_length, 13)`

The sliding window parameters were:
- `seq_length = 30` (approximately 0.5-1 second of motion at typical frame rates)
- `stride = 15` (50% overlap between consecutive sequences)

### ⚠️ BEWARE: Data Leakage in Sequence Splitting

A critical discovery was made regarding how sequences are split into train/validation/test sets. **Improper splitting leads to severe data leakage and artificially inflated performance metrics.**

#### The Problem

When sequences are created with overlap (`stride < seq_length`) and then **randomly split** across train/val/test sets, the same underlying frames appear in multiple splits:

| Split | Sequence | Frames | Overlap |
|-------|----------|--------|---------|
| Training | Seq 1 | 0-59 | - |
| Test | Seq 2 | 1-60 | 98.3% |
| Validation | Seq 3 | 2-61 | 96.7% |

This causes:
- ❌ Validation loss lower than training loss 
- ❌ Models that fail to generalize to new, unseen videos
- ❌ Overly optimistic performance metrics

### Leaky metrics

| Metric | Value |
|--------|-------|
| Test Loss | 0.000588 |
| Test MAE | 0.01396 |
| Test R² | 0.970 |


#### The Solution

To prevent data leakage,split by video file, not by sequence
